# RAG with Amazon Bedrock Managed Knowledge Base

This notebook demonstrates how to use a **Managed Knowledge Base** for Retrieval Augmented Generation (RAG).

Unlike the vector-based approach (which requires OpenSearch Serverless), a Managed Knowledge Base handles embedding, storage, and retrieval automatically — no external vector store needed.

## Prerequisites

1. Deploy `template_managed.yml` (or create a Managed KB via the Bedrock console)
2. Upload documents to the S3 bucket
3. Start an ingestion job to sync the data
4. `boto3 >= 1.41` (for managed search), `>= 1.43` (for agentic retrieval)

In [1]:
%pip install --upgrade boto3 botocore

In [2]:
import boto3
import json

# REPLACE with your Managed Knowledge Base ID (from template_managed.yml stack output)
KNOWLEDGE_BASE_ID = ""  # e.g., "5UDVRO91CQ"
REGION = "us-west-2"

client = boto3.client('bedrock-agent-runtime', region_name=REGION)
print(f"boto3 version: {boto3.__version__}")
print(f"Knowledge Base ID: {KNOWLEDGE_BASE_ID}")

boto3 version: 1.43.36
Knowledge Base ID: 5UDVRO91CQ


## 1. Basic Retrieval with Managed Search

For managed KBs, use `managedSearchConfiguration` (not `vectorSearchConfiguration`).
Reranking is enabled by default using a service-managed model.

In [3]:
question = "Where does Robin live?"

response = client.retrieve(
    knowledgeBaseId=KNOWLEDGE_BASE_ID,
    retrievalQuery={'text': question},
    retrievalConfiguration={
        'managedSearchConfiguration': {
            'numberOfResults': 5
        }
    }
)

print(f"Question: {question}")
print(f"Results: {len(response['retrievalResults'])}\n")

for i, result in enumerate(response['retrievalResults'], 1):
    score = result.get('score', 'N/A')
    content = result['content']['text'][:200]
    print(f"[{i}] Score: {score}")
    print(f"    {content}...\n")

Question: Where does Robin live?
Results: 1

[1] Score: 1.0
    My name is Robin. I live in Hyderabad. I work as a software engineer and enjoy playing cricket on weekends.


## 2. Minimal Request (API Auto-Detects)

You can also send no `retrievalConfiguration` at all — the API auto-detects the KB type and applies the correct search.

In [4]:
response = client.retrieve(
    knowledgeBaseId=KNOWLEDGE_BASE_ID,
    retrievalQuery={'text': 'What are the benefits of managed knowledge bases?'}
)

print(f"Results (no config): {len(response['retrievalResults'])}")
for i, result in enumerate(response['retrievalResults'][:3], 1):
    print(f"[{i}] {result['content']['text'][:150]}...")

Results (no config): 1
[1] My name is Robin. I live in Hyderabad. I work as a software engineer and enjoy playing cricket on weekends.


## 3. Reranking Options

Managed KBs use a service-managed reranker by default. You can control this:
- `MANAGED` (default) — automatic reranking
- `NONE` — disable reranking
- `CUSTOM` — use your own Bedrock reranking model

In [5]:
# Disable reranking
response_no_rerank = client.retrieve(
    knowledgeBaseId=KNOWLEDGE_BASE_ID,
    retrievalQuery={'text': question},
    retrievalConfiguration={
        'managedSearchConfiguration': {
            'numberOfResults': 5,
            'rerankingModelType': 'NONE'
        }
    }
)

print(f"Results (no reranking): {len(response_no_rerank['retrievalResults'])}")
for i, result in enumerate(response_no_rerank['retrievalResults'][:3], 1):
    print(f"[{i}] Score: {result.get('score', 'N/A')} — {result['content']['text'][:100]}...")

Results (no reranking): 1
[1] Score: 1.0 — My name is Robin. I live in Hyderabad. I work as a software engineer and enjoy playing cricket on weekends.


## 4. Agentic Retrieval (Advanced)

For complex queries that benefit from query decomposition and iterative retrieval, use `AgenticRetrieveStream`.

This API:
- Decomposes complex queries into sub-queries
- Retrieves iteratively across the knowledge base
- Reranks results using a managed model
- Optionally generates a cited answer

**Requires boto3 >= 1.43**

In [6]:
try:
    agentic_response = client.agentic_retrieve_stream(
        messages=[{"content": {"text": question}, "role": "user"}],
        retrievers=[{
            "configuration": {
                "knowledgeBase": {
                    "knowledgeBaseId": KNOWLEDGE_BASE_ID,
                    "retrievalOverrides": {"maxNumberOfResults": 5}
                }
            }
        }],
        agenticRetrieveConfiguration={
            "foundationModelType": "MANAGED",
            "rerankingModelType": "MANAGED"
        },
        generateResponse=False
    )

    print("Agentic Retrieval Results:\n")
    for event in agentic_response.get("stream", []):
        if "result" in event:
            results = event["result"].get("results", [])
            for i, r in enumerate(results, 1):
                content = r.get("content", {}).get("text", "")[:150]
                source = r.get("metadata", {}).get("_source_uri", "Unknown")
                print(f"[{i}] {content}...")
                print(f"    Source: {source}\n")

except Exception as e:
    print(f"AgenticRetrieveStream not available: {e}")
    print("Requires boto3 >= 1.43. Run: pip install --upgrade boto3")

## 5. Agentic Retrieval with Response Generation

Set `generateResponse=True` to get a synthesized answer with citations.

In [7]:
try:
    agentic_response = client.agentic_retrieve_stream(
        messages=[{"content": {"text": question}, "role": "user"}],
        retrievers=[{
            "configuration": {
                "knowledgeBase": {
                    "knowledgeBaseId": KNOWLEDGE_BASE_ID,
                    "retrievalOverrides": {"maxNumberOfResults": 5}
                }
            }
        }],
        agenticRetrieveConfiguration={
            "foundationModelType": "MANAGED",
            "rerankingModelType": "MANAGED"
        },
        generateResponse=True
    )

    print("Generated Answer:\n")
    for event in agentic_response.get("stream", []):
        if "result" in event and "generatedResponse" in event["result"]:
            answer = event["result"]["generatedResponse"].get("answer", "")
            print(answer)
        elif "responseEvent" in event:
            print(event["responseEvent"].get("text", ""), end="")

except Exception as e:
    print(f"AgenticRetrieveStream not available: {e}")
    print("Requires boto3 >= 1.43. Run: pip install --upgrade boto3")

## Comparison: Vector KB vs Managed KB

| | Vector KB (template.yml) | Managed KB (template_managed.yml) |
|---|---|---|
| Vector store | You manage (OpenSearch Serverless) | Bedrock manages |
| Embedding model | You choose (Titan Embed) | Service-managed |
| Retrieval config | `vectorSearchConfiguration` | `managedSearchConfiguration` |
| Reranking | Manual setup | Managed by default |
| Agentic retrieval | Not available | Available |
| Cost | KB + AOSS (min 2 OCU) | KB only |
| RetrieveAndGenerate | Supported | Not supported (use AgenticRetrieveStream) |

## Resources

- [Build a Managed Knowledge Base](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-build-managed.html)
- [Query a Knowledge Base (Retrieve API)](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-test-retrieve.html)
- [Agentic Retrieval](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-test-agentic.html)